# 01: 量子ビットの番号付けとケット表記

本プロジェクトでは、物理ノート（`notes/`）と Qiskit の notebook（`notebooks/`）で量子ビットの記法が異なる。ここではその違いを整理し、**ラベルを一貫して付け替えれば物理的な結果は変わらない**ことを回路で確認する。

他の notebook を読む前に、この点を押さえておくと混乱を避けられる。

In [1]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np
import matplotlib.pyplot as plt

## 1. インデックスの違い

ノートでは量子ビットを $q_1, q_2, q_3$ と **1 始まり**で番号付けしている。一方、Qiskit では **0 始まり**なので $q_0, q_1, q_2$ となる。

| ノート（1 始まり） | Notebook / Qiskit（0 始まり） |
|---|---|
| $q_1$ | $q_0$ |
| $q_2$ | $q_1$ |
| $q_3$ | $q_2$ |

ゲートの添字も同様に、例えばノートの $\text{CNOT}_{12}$（$q_1$ 制御、$q_2$ 標的）は、notebook では $\text{CNOT}_{01}$（$q_0$ 制御、$q_1$ 標的）と記す。

## 2. ケット表記・bitstring・状態ベクトル配列の違い

ノートの回路図も Qiskit で生成しているため、回路図のラベルはどちらも上から $q_0, q_1, q_2$ で同じである。ただし、ノートの本文中の数式では $q_1, q_2, q_3$（1 始まり）で記述しているため、回路図と本文でインデックスが 1 つずれる点に注意。

特に混乱しやすいのは、以下の3つの場面である。

### ケット表記

状態をケット $\vert\cdots\rangle$ で書くとき、ビットの並び順が異なる。

- **ノート本文（教科書規約）:** $\vert q_1\, q_2\, q_3\rangle$ と、番号順（1 始まり）で左から並べる
- **Qiskit（リトルエンディアン規約）:** $\vert q_2\, q_1\, q_0\rangle$ と、$q_0$ を**右端**に置く

例えば、回路図で一番上のワイヤー（$q_0$）が $1$、残りが $0$ の状態は：

| 規約 | ケット表記 |
|---|---|
| ノート本文 | $\vert 100\rangle$（$q_1 = 1, q_2 = 0, q_3 = 0$） |
| Qiskit | $\vert 001\rangle$（$q_2 = 0, q_1 = 0, q_0 = 1$） |

同じ物理状態なのに、ビットの並び順が逆になる。

### bitstring（測定結果の文字列）

Qiskit の測定結果は文字列 `"abc"` で返されるが、ここでも bit 0（$q_0$）が**右端**に来る。例えば `"01"` は $q_1 = 0, q_0 = 1$ を意味する。

### 状態ベクトル配列の添字

Qiskit の `Statevector` では、配列のインデックス $i$ が計算基底 $\vert i\rangle$ に対応する。$i$ の二進数表現で bit 0（最下位ビット）が $q_0$ の値になる。例えば 2量子ビットでインデックス 1（二進数 `01`）は $q_1 = 0, q_0 = 1$ である。bitstring と同じ並び順である。

## 3. ラベルを一貫して付け替えれば物理は変わらない

ノートとQiskit で記法が異なるが、物理的な結果に影響はあるのか？

答えは**ラベルを一貫して付け替える限り、物理は変わらない**である。つまり、すべてのゲート・初期状態・測定を同じ対応関係で読み替えれば、測定結果の分布は同じになる。

以下では、**同一の回路**をノートの規約と Qiskit の規約でそれぞれ読み、同じ物理であることを確認する。

### 例 1: 1量子ビットへの $X$ ゲート

2量子ビットの回路で、上のワイヤーに $X$ を掛ける。この **1つの回路** をノートの規約と Qiskit の規約で読み比べる。

![X on q_0](../images/01_x_on_q0.png)

- **Qiskit:** 上のワイヤーは $q_0$。$X$ が $q_0$ に掛かり、状態は $q_0 = 1, q_1 = 0$。ケット表記は $\vert q_1\, q_0\rangle = \vert 01\rangle$。
- **ノート:** 上のワイヤーは $q_1$。$X$ が $q_1$ に掛かり、状態は $q_1 = 1, q_2 = 0$。ケット表記は $\vert q_1\, q_2\rangle = \vert 10\rangle$。

ケット表記は $\vert 01\rangle$ と $\vert 10\rangle$ で異なるが、回路は同一であり、「上のワイヤーを測定すれば必ず 1」という物理は同じである。

In [ ]:
# 上のワイヤーに X を掛ける回路（1つだけ）
qc = QuantumCircuit(2)
qc.x(0)  # Qiskit では上のワイヤーが q_0

# 状態ベクトルを確認
sv = Statevector.from_instruction(qc)

print("状態ベクトル配列:", np.array(sv).round(4))
print()
for i, amp in enumerate(sv):
    if abs(amp) > 1e-10:
        q0 = i & 1       # Qiskit の q_0（上のワイヤー）
        q1 = (i >> 1) & 1 # Qiskit の q_1（下のワイヤー）
        print(f"  配列インデックス {i} (二進数 {i:02b})")
        print(f"    Qiskit: q_0={q0}, q_1={q1} → ケット |q_1 q_0⟩ = |{q1}{q0}⟩")
        print(f"    ノート: q_1={q0}, q_2={q1} → ケット |q_1 q_2⟩ = |{q0}{q1}⟩")

print()
print("→ Qiskit では |01⟩、ノートでは |10⟩ と書くが、同じ回路の同じ状態。")
print("   上のワイヤーを測定すれば必ず 1 が出る、という物理は変わらない。")

### 例 2: ベル状態の生成

上のワイヤーに $H$、上から下へ CNOT を掛ける。この **1つの回路** を両方の規約で読む。

![ベル状態回路](../images/01_bell.png)

- **Qiskit:** $H$ on $q_0$, $\text{CNOT}_{01}$ → ケット $(\vert 00\rangle + \vert 11\rangle)/\sqrt{2}$（$q_0$ が右）
- **ノート:** $H$ on $q_1$, $\text{CNOT}_{12}$ → ケット $(\vert 00\rangle + \vert 11\rangle)/\sqrt{2}$（$q_1$ が左）

この場合はケット表記も同じ $\vert 00\rangle + \vert 11\rangle$ になる。これは偶然ではなく、$\vert 00\rangle$ と $\vert 11\rangle$ はビットを逆順に並べても同じ文字列だからである。

重要なのはケット表記の見た目が一致するかではなく、「2つのワイヤーの測定結果が常に一致する」という物理が同じであることである。

In [ ]:
# ベル状態の回路（1つだけ）
qc_bell = QuantumCircuit(2)
qc_bell.h(0)      # 上のワイヤーに H
qc_bell.cx(0, 1)  # 上から下へ CNOT

# 状態ベクトルを確認
sv_bell = Statevector.from_instruction(qc_bell)

print("状態ベクトル配列:", np.array(sv_bell).round(4))
print()
print("非ゼロの振幅:")
for i, amp in enumerate(sv_bell):
    if abs(amp) > 1e-10:
        q0 = i & 1
        q1 = (i >> 1) & 1
        print(f"  配列インデックス {i} (二進数 {i:02b})")
        print(f"    Qiskit: q_0={q0}, q_1={q1} → ケット |{q1}{q0}⟩")
        print(f"    ノート: q_1={q0}, q_2={q1} → ケット |{q0}{q1}⟩")

print()
print("→ どちらの規約でも |00⟩ + |11⟩ と書ける（この場合は逆順でも同じ文字列）。")
print("   物理的に重要なのは「2つのワイヤーの測定結果が常に一致する」ことであり、")
print("   これはラベルの付け方によらない。")

## 4. まとめ

| 項目 | ノート本文（教科書規約） | Notebook / 回路図（Qiskit） |
|---|---|---|
| インデックス | 1 始まり（$q_1, q_2, \ldots$） | 0 始まり（$q_0, q_1, \ldots$） |
| ケット表記 | $\vert q_1\, q_2\, q_3\rangle$（番号順、左から） | $\vert q_2\, q_1\, q_0\rangle$（$q_0$ が右端） |
| 回路図のラベル | Qiskit 生成なので $q_0, q_1, \ldots$（本文とずれる） | $q_0, q_1, \ldots$ |
| 回路図の接続関係 | 同じ | 同じ |
| 物理的な結果 | 同じ | 同じ |

ラベルを一貫して付け替える限り、物理的な結果は変わらない。重要なのは「どのワイヤーにどのゲートが掛かり、どのワイヤーを測定するか」という接続関係だけであり、それを $q_0$ と呼ぶか $q_1$ と呼ぶかは結果に影響しない。

以降の notebook（02, 03, 05, 06）では、この点を気にせず回路の動作に集中してよい。

## 5. 注意：測定結果を整数として解釈するとき

ラベルの付け替えで物理は変わらないが、**測定結果を整数として読むときにはビット順の規約を揃える必要がある**。

量子位相推定（QPE、ノート05）では、各ワイヤーに掛かる $U_a$ の冪が異なる：

- 一番上のワイヤー（$q_0$）が $U_a^{2^{n-1}}$ を制御 → **最上位ビット（MSB）**
- 一番下のワイヤー（$q_{n-1}$）が $U_a^1$ を制御 → **最下位ビット（LSB）**

測定後、位相を $\varphi = \tilde{s}/2^n$ として復元するには、各ワイヤーの測定値を正しいビット位置に対応させて整数 $\tilde{s}$ を組み立てなければならない。

例えば $n = 3$ で、上のワイヤーだけが 1（残りは 0）のとき：

| 読み方 | 整数値 | 正しいか |
|---|---|---|
| 教科書的に MSB から読む: $100_2$ | $\tilde{s} = 4$ | 正しい |
| Qiskit の配列インデックスをそのまま使う: index = 1 | $\tilde{s} = 1$ | **間違い** |

Qiskit のリトルエンディアン規約では $q_0$ が LSB（最下位ビット）だが、この回路では $q_0$ が MSB（最上位ビット）を担っている。そのため、配列インデックスをそのまま整数 $\tilde{s}$ として使うとビット順が逆になり、間違った位相を計算してしまう。

![QPE 整数解釈の例](../images/01_qpe_example.png)

In [ ]:
# QPE での整数解釈の例（n=3 ビット）
# 上のワイヤー (q_0) だけが 1、残りは 0 の状態を作る

n = 3
qc_qpe = QuantumCircuit(n)
qc_qpe.x(0)  # 上のワイヤー (q_0) を 1 にする

# 状態ベクトルで整数解釈の違いを確認
sv_qpe = Statevector.from_instruction(qc_qpe)

for i, amp in enumerate(sv_qpe):
    if abs(amp) > 1e-10:
        bits = [(i >> k) & 1 for k in range(n)]
        
        # 方法 1: 配列インデックスをそのまま使う（間違い）
        s_wrong = i
        
        # 方法 2: q_0 を MSB として正しく変換
        s_correct = sum(bits[k] * 2 ** (n - 1 - k) for k in range(n))
        
        print(f"配列インデックス: {i} (二進数 {i:03b})")
        print(f"各ワイヤーの値: q_0={bits[0]}, q_1={bits[1]}, q_2={bits[2]}")
        print()
        print(f"  ✗ インデックスをそのまま s とする: s = {s_wrong} → φ = {s_wrong}/{2**n} = {s_wrong/2**n}")
        print(f"  ✓ q_0 を MSB として変換:          s = {s_correct} → φ = {s_correct}/{2**n} = {s_correct/2**n}")
        print()
        print(f"QPE の回路では q_0 が U^{2**(n-1)} を制御するので、q_0 は MSB。")
        print(f"正しい整数値を得るには、ビット順を明示的に変換する必要がある。")

つまり、「どのワイヤーが 0 でどのワイヤーが 1 か」という物理状態はラベルによらず同じだが、その測定値を**1つの整数にまとめる**段階でビット順の規約が必要になる。notebook 05, 06 のコードではこの変換を明示的に行っている。

物理は変わらない。しかし、整数としての解釈は規約に依存する——この区別を意識しておくことが重要である。